# Fase 4: Matriz GEMSES integrada y corregida

**Autor:** Carlos Perez Perez | **MIA 303** | **17/05/2026**

---

Este notebook hace dos cosas:

1. **Integra** las detecciones de la Fase 2-NLP (filtradas con NegEx) con las detecciones
   estructurales de Fase 3 (estancia prolongada). El resultado es la primera Matriz de
   Priorizacion GEMSES "completa", con eventos clinicos y administrativos sobre la misma
   base de 2,000 hospitalizaciones.

2. **Aplica la correccion institucional** de los 33 eventos del Anexo 2 cuyo Impacto
   reportado no satisface la formula GEMSES (0.40c + 0.20d + 0.15e + 0.20f). La tesis
   usa el valor recalculado y lo propone como mejora institucional para EsSalud.


## 1. Cargar las tres fuentes

In [1]:
import duckdb, pandas as pd
from pathlib import Path
PATH_WORK = Path(r"C:\MIMIC\tesis")

# 1.1 NegEx-filtradas (Fase 2)
det_nlp = pd.read_csv(PATH_WORK / "fase2_detecciones_filtradas_negex.csv", encoding='utf-8-sig')
det_nlp_afirm = det_nlp[det_nlp['contexto']=='AFIRMADO'].copy()
print(f"Detecciones NLP afirmadas (Fase 2): {len(det_nlp_afirm)}")

# 1.2 Estancia prolongada (Fase 3): solo las que caen dentro de las 2,000 hadm_id de la muestra NLP
hadm_muestra = set(det_nlp['hadm_id'].unique())
los_pos = pd.read_csv(PATH_WORK / "fase3_estancia_prolongada_positivas.csv", encoding='utf-8-sig')
los_en_muestra = los_pos[los_pos['hadm_id'].isin(hadm_muestra)].copy()
print(f"Estancia prolongada en la muestra (Fase 3): {len(los_en_muestra)}")

# 1.3 Anexo 2 con sus dos valores de impacto
con = duckdb.connect(r"C:\BASE DATOS\db\gemses.db", read_only=True)
anexo2 = con.execute(
    "SELECT id_evento, evento, naturaleza, impacto, impacto_formula, verificacion "
    "FROM anexo2_eventos_adversos").df()
con.close()
anexo2['impacto_corregido'] = anexo2['impacto_formula']
print(f"Anexo 2 cargado: {len(anexo2)} eventos")
print(f"  - sin cambio (OK)        : {(anexo2['verificacion']=='OK').sum()}")
print(f"  - con correccion aplicada: {(anexo2['verificacion']=='REVISAR').sum()}")


Detecciones NLP afirmadas (Fase 2): 1172


Estancia prolongada en la muestra (Fase 3): 327
Anexo 2 cargado: 231 eventos
  - sin cambio (OK)        : 198
  - con correccion aplicada: 33


## 2. Unificar las dos fuentes de deteccion

Cada fuente se normaliza al mismo esquema: una fila por (hadm_id, id_evento)
detectado. Se concatenan.

In [2]:
# normalizar NegEx
det_nlp_norm = det_nlp_afirm[['subject_id','hadm_id','id_evento','evento','naturaleza','confianza']].copy()
det_nlp_norm['fuente'] = 'NLP_NegEx'
det_nlp_norm = det_nlp_norm.drop_duplicates(['hadm_id','id_evento'])

# normalizar estancia prolongada
los_norm = los_en_muestra[['subject_id','hadm_id','id_evento','evento','naturaleza','confianza']].copy()
los_norm['fuente'] = 'LOS_DRG_P75'
los_norm = los_norm.drop_duplicates(['hadm_id','id_evento'])

detecciones = pd.concat([det_nlp_norm, los_norm], ignore_index=True)
print(f"Detecciones totales integradas: {len(detecciones)}")
print(f"  fuentes: {detecciones['fuente'].value_counts().to_dict()}")
print(f"  hadm_id unicos: {detecciones['hadm_id'].nunique()}")
print(f"  id_evento unicos: {detecciones['id_evento'].nunique()}")


Detecciones totales integradas: 1499
  fuentes: {'NLP_NegEx': 1172, 'LOS_DRG_P75': 327}
  hadm_id unicos: 935
  id_evento unicos: 33


## 3. Calcular la Matriz GEMSES con Impacto corregido

Aplicamos la formula oficial usando el `impacto_corregido` (valor recalculado de
los 33 eventos con la formula GEMSES, igual al oficial en los otros 198).

In [3]:
frec = (detecciones.groupby(['id_evento','evento','naturaleza','confianza','fuente'])
        .size().reset_index(name='A_frecuencia'))
# si un evento tiene mas de una fuente, lo consolidamos sumando
frec = (detecciones.groupby(['id_evento','evento','naturaleza'])
        .agg(A_frecuencia=('hadm_id','count'),
             fuentes=('fuente', lambda x: '+'.join(sorted(set(x)))))
        .reset_index())

matriz = frec.merge(
    anexo2[['id_evento','impacto','impacto_corregido','verificacion']],
    on='id_evento', how='left')
matriz = matriz.rename(columns={'impacto':'G_impacto_directiva',
                                'impacto_corregido':'G_impacto_corregido'})
total = matriz['A_frecuencia'].sum()
matriz['B_indice_frecuencia'] = (matriz['A_frecuencia'] / total).round(4)
matriz['H_riesgo']            = (matriz['B_indice_frecuencia'] * matriz['G_impacto_corregido']).round(4)
sH = matriz['H_riesgo'].sum()
matriz['I_indice_impacto']    = (matriz['H_riesgo'] / sH).round(4)
matriz['J_nivel_gestion']     = (matriz['I_indice_impacto'] * 100).round(2)
q2, q3 = matriz['J_nivel_gestion'].quantile([0.50, 0.75])
matriz['nivel_gestion'] = matriz['J_nivel_gestion'].apply(
    lambda j: 'ROJO (Director)' if j>=q3 else ('AMARILLO (Departamento)' if j>=q2 else 'VERDE (Servicio)'))

print(f"Eventos en la matriz integrada: {len(matriz)}")
print(f"Total de detecciones (suma A) : {total}")


Eventos en la matriz integrada: 33
Total de detecciones (suma A) : 1499


## 4. Top 15 eventos por Riesgo (H) — Matriz GEMSES integrada y corregida

In [4]:
cols = ['id_evento','evento','naturaleza','fuentes','A_frecuencia','B_indice_frecuencia',
        'G_impacto_directiva','G_impacto_corregido','H_riesgo','J_nivel_gestion','nivel_gestion','verificacion']
top = matriz.sort_values('H_riesgo', ascending=False)[cols].head(15)
top


,id_evento,evento,naturaleza,fuentes,A_frecuencia,B_indice_frecuencia,G_impacto_directiva,G_impacto_corregido,H_riesgo,J_nivel_gestion,nivel_gestion,verificacion
9,1007,Estancia prolongada,GESTION DE LA ORGANIZACION,LOS_DRG_P75,327,0.2181,7.35,7.95,1.7339,37.73,ROJO (Director),REVISAR
5,602,Infección del tracto urinario,INFECCIÓN ASOCIADA A LA ATENCIÓN,NLP_NegEx,195,0.1301,4.50,4.50,0.5854,12.74,ROJO (Director),OK
25,7031,Diarrea asociada a antimicrobianos,MEDICACIÓN,NLP_NegEx,88,0.0587,5.70,5.70,0.3346,7.28,ROJO (Director),OK
31,8032,Trombosis venosa,PROCEDIMIENTO,NLP_NegEx,113,0.0754,3.30,3.30,0.2488,5.41,ROJO (Director),OK
2,202,Caída de paciente,CUIDADO DEL PACIENTE,NLP_NegEx,250,0.1668,1.35,1.35,0.2252,4.90,ROJO (Director),OK
4,601,Infección del torrente sanguíneo (ITS),INFECCIÓN ASOCIADA A LA ATENCIÓN,NLP_NegEx,75,0.0500,4.50,4.50,0.2250,4.90,ROJO (Director),OK
18,6030,Osteomielitis,INFECCIÓN ASOCIADA A LA ATENCIÓN,NLP_NegEx,29,0.0193,8.55,8.55,0.1650,3.59,ROJO (Director),OK
17,6022,Infección de la piel,INFECCIÓN ASOCIADA A LA ATENCIÓN,NLP_NegEx,102,0.0680,2.05,2.05,0.1394,3.03,ROJO (Director),OK
7,705,Dosis o frecuencia errónea,MEDICACIÓN,NLP_NegEx,50,0.0334,3.70,3.70,0.1236,2.69,ROJO (Director),OK
27,8016,Neumotórax,PROCEDIMIENTO,NLP_NegEx,40,0.0267,4.30,4.30,0.1148,2.50,AMARILLO (Departamento),OK


## 5. Comparacion: ¿que cambia con la correccion institucional?

Cuanto cambian el ranking y los niveles si comparamos usando el impacto de la
directiva vs el corregido por formula.

In [5]:
# duplicar matriz para el escenario "directiva tal cual"
mat_dir = matriz.copy()
mat_dir['H_dir'] = (mat_dir['B_indice_frecuencia'] * mat_dir['G_impacto_directiva']).round(4)
sH_dir = mat_dir['H_dir'].sum()
mat_dir['I_dir'] = (mat_dir['H_dir'] / sH_dir).round(4)
mat_dir['J_dir'] = (mat_dir['I_dir'] * 100).round(2)
q2d, q3d = mat_dir['J_dir'].quantile([0.50, 0.75])
mat_dir['nivel_dir'] = mat_dir['J_dir'].apply(
    lambda j: 'ROJO' if j>=q3d else ('AMARILLO' if j>=q2d else 'VERDE'))

comp = matriz[['id_evento','evento','verificacion']].merge(
    mat_dir[['id_evento','nivel_dir']], on='id_evento').merge(
    matriz[['id_evento','nivel_gestion']], on='id_evento')
comp.columns = ['id_evento','evento','verificacion','nivel_con_directiva','nivel_con_correccion']
comp['cambio'] = comp['nivel_con_directiva'].str.split().str[0] != comp['nivel_con_correccion'].str.split().str[0]
n_cambio = comp['cambio'].sum()
print(f"Eventos cuyo nivel de gestion cambia con la correccion: {n_cambio} de {len(comp)}")
if n_cambio > 0:
    print(f"\nDetalle de los cambios:")
    print(comp[comp['cambio']].to_string(index=False))


Eventos cuyo nivel de gestion cambia con la correccion: 0 de 33


## 6. Guardar la matriz integrada final

In [6]:
out_matriz = PATH_WORK / "fase4_matriz_gemses_integrada.csv"
matriz_out = matriz[cols].copy()
matriz_out.to_csv(out_matriz, index=False, encoding='utf-8-sig')
print(f"Guardado: {out_matriz.name}  ({len(matriz_out)} eventos)")

out_det = PATH_WORK / "fase4_detecciones_integradas.csv"
detecciones.to_csv(out_det, index=False, encoding='utf-8-sig')
print(f"Guardado: {out_det.name}  ({len(detecciones)} detecciones)")


Guardado: fase4_matriz_gemses_integrada.csv  (33 eventos)
Guardado: fase4_detecciones_integradas.csv  (1499 detecciones)


## 7. Resumen para el Capitulo III

- **Matriz GEMSES integrada**: combina detecciones de dos fuentes complementarias
  (texto via NegEx + datos estructurales LOS-DRG).
- **Correccion institucional aplicada**: los 33 eventos del Anexo 2 con Impacto
  inconsistente fueron recalculados segun la formula GEMSES oficial. Se documenta
  como propuesta de mejora institucional para EsSalud.
- **Trazabilidad**: la matriz mantiene columna `G_impacto_directiva` (valor oficial)
  junto a `G_impacto_corregido` (valor usado en los calculos), para que cualquier
  auditor pueda verificar la diferencia evento por evento.
